In [1]:
!pip install transformers datasets

In [2]:
from datasets import load_dataset

raw_dataset = load_dataset('squad')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [3]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [4]:
raw_dataset['train'][1]['title']

'University_of_Notre_Dame'

In [5]:
raw_dataset['train'][1]['context']

'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.'

In [6]:
raw_dataset['train'][1]['question']

'What is in front of the Notre Dame Main Building?'

In [7]:
raw_dataset['train'][1]['answers']

{'text': ['a copper statue of Christ'], 'answer_start': [188]}

In [8]:
raw_dataset['train'].filter(lambda x: len(x['answers']['text']) != 1)


Filter:   0%|          | 0/87599 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 0
})

In [9]:
raw_dataset['validation'][2]['answers']

{'text': ['Santa Clara, California',
  "Levi's Stadium",
  "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."],
 'answer_start': [403, 355, 355]}

In [10]:
raw_dataset['validation'][2]['context']

'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.'

In [11]:
raw_dataset['validation'][2]['question']

'Where did Super Bowl 50 take place?'

In [12]:
raw_dataset['validation'][0]['answers']

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

In [13]:
from transformers import AutoTokenizer

In [14]:
checkpoint = 'distilbert-base-cased'

In [15]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
context = raw_dataset['train'][1]['context']
question = raw_dataset['train'][1]['question']

inputs = tokenizer(question, context)
tokenizer.decode(inputs['input_ids'])

'[CLS] What is in front of the Notre Dame Main Building? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building \' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive ( and in a direct line that connects through 3 statues and the Gold Dome ), is a simple, modern stone statue of Mary. [SEP]'

In [17]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation='only_second',
    stride=50,
    return_overflowing_tokens=True)

for ids in inputs['input_ids']:
  print(tokenizer.decode(ids))

[CLS] What is in front of the Notre Dame Main Building? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the G [SEP]
[CLS] What is in front of the Notre Dame Main Building? [SEP] facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernade [SEP]
[CLS] What is in front of the Notre Dame Main Building? [SEP] of the Sacred Heart. Immediately behind the basilica is the Grotto

In [18]:
for key in inputs.keys():
    print(key)

input_ids
token_type_ids
attention_mask
overflow_to_sample_mapping


In [19]:
keys = list(inputs.keys())
keys

['input_ids', 'token_type_ids', 'attention_mask', 'overflow_to_sample_mapping']

In [20]:
inputs['overflow_to_sample_mapping']

[0, 0, 0, 0]

In [21]:
inputs = tokenizer(
    raw_dataset['train'][:3]['question'],
    raw_dataset['train'][:3]['context'],
    max_length=100,
    truncation='only_second',
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True)

In [22]:
inputs['overflow_to_sample_mapping']

[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]

In [23]:
for ids in inputs['input_ids']:
  print(tokenizer.decode(ids))

[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basi [SEP]
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin [SEP]
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Next to the Main Building is the

In [24]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation='only_second',
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True)

list(inputs.keys())

['input_ids',
 'token_type_ids',
 'attention_mask',
 'offset_mapping',
 'overflow_to_sample_mapping']

In [25]:
inputs['offset_mapping']

[[(0, 0),
  (0, 4),
  (5, 7),
  (8, 10),
  (11, 16),
  (17, 19),
  (20, 23),
  (24, 29),
  (30, 34),
  (35, 39),
  (40, 48),
  (48, 49),
  (0, 0),
  (0, 13),
  (13, 15),
  (15, 16),
  (17, 20),
  (21, 27),
  (28, 31),
  (32, 33),
  (34, 42),
  (43, 52),
  (52, 53),
  (54, 56),
  (56, 58),
  (59, 62),
  (63, 67),
  (68, 76),
  (76, 77),
  (77, 78),
  (79, 83),
  (84, 88),
  (89, 91),
  (92, 93),
  (94, 100),
  (101, 107),
  (108, 110),
  (111, 114),
  (115, 121),
  (122, 126),
  (126, 127),
  (128, 139),
  (140, 142),
  (143, 148),
  (149, 151),
  (152, 155),
  (156, 160),
  (161, 169),
  (170, 173),
  (174, 180),
  (181, 183),
  (183, 184),
  (185, 187),
  (188, 189),
  (190, 196),
  (197, 203),
  (204, 206),
  (207, 213),
  (214, 218),
  (219, 223),
  (224, 226),
  (226, 229),
  (229, 232),
  (233, 237),
  (238, 241),
  (242, 248),
  (249, 250),
  (250, 251),
  (251, 254),
  (254, 256),
  (257, 259),
  (260, 262),
  (263, 264),
  (264, 265),
  (265, 268),
  (268, 269),
  (269, 270),
 

In [26]:
len(inputs['offset_mapping'])

4

In [27]:
len(inputs['offset_mapping'][3])

69

In [28]:
print(inputs.sequence_ids(0))

[None, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, None, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, None]


In [29]:
answer = raw_dataset['train'][1]['answers']

In [30]:
answer

{'text': ['a copper statue of Christ'], 'answer_start': [188]}

In [31]:
type(inputs.sequence_ids(0))

list

In [32]:
sequence_ids = inputs.sequence_ids(0)

In [33]:
ctx_start = sequence_ids.index(1)

In [34]:
ctx_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1

In [35]:
ctx_start, ctx_end

(13, 98)

In [36]:
ans_char_start = answer['answer_start'][0]
ans_char_end = ans_char_start + len(answer['text'][0])

In [37]:
ans_char_start, ans_char_end

(188, 213)

In [38]:
offset = inputs['offset_mapping'][0]

In [39]:
start_idx = 0
end_idx = 0

In [40]:
if offset[ctx_start][0] > ans_char_start or offset[ctx_end][1] < ans_char_end:
  print('target is (0, 0)')
else:
  i = ctx_start
  for start_end_char in offset[ctx_start:]:
    start, end = start_end_char
    if start == ans_char_start:
      start_idx = i

    if end == ans_char_end:
      end_idx = i
      break

    i += 1

start_idx, end_idx

(53, 57)

In [41]:
input_ids = inputs['input_ids'][0]

In [42]:
input_ids[start_idx:end_idx+1]

[170, 7335, 5921, 1104, 4028]

In [43]:
tokenizer.decode(input_ids[start_idx:end_idx+1])

'a copper statue of Christ'

In [44]:
def find_answer_token_idx(
    ctx_start,
    ctx_end,
    ans_char_start,
    ans_char_end,
    offset
):
  start_idx = 0
  end_idx = 0

  if offset[ctx_start][0] > ans_char_start or offset[ctx_end][1] < ans_char_end:
    pass
  else:
    i = ctx_start
    for start_end_char in offset[ctx_start:]:
      start, end = start_end_char
      if start == ans_char_start:
        start_idx = i

      if end == ans_char_end:
        end_idx = i
        break

      i += 1
  return start_idx, end_idx

In [45]:
start_idxs = []
end_idxs = []

for i, offset in enumerate(inputs['offset_mapping']):
  sequence_ids = inputs.sequence_ids(i)

  ctx_start = sequence_ids.index(1)
  ctx_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1

  start_idx, end_idx = find_answer_token_idx(
      ctx_start,
      ctx_end,
      ans_char_start,
      ans_char_end,
      offset
    )

  start_idxs.append(start_idx)
  end_idxs.append(end_idx)

start_idxs, end_idxs

([53, 17, 0, 0], [57, 21, 0, 0])

In [46]:
for q in raw_dataset['train']['question'][:1000]:
  if q.strip() != q:
    print(q)

In what city and state did Beyonce  grow up? 
 The album, Dangerously in Love  achieved what spot on the Billboard Top 100 chart?
Which song did Beyonce sing at the first couple's inaugural ball? 
What event did Beyoncé perform at one month after Obama's inauguration? 
Where was the album released? 
What movie influenced Beyonce towards empowerment themes? 


In [47]:
max_length = 384
stride = 128

def tokenize_fn_train(batch):
  questions = [q.strip() for q in batch['question']]

  inputs = tokenizer(
    questions,
    batch['context'],
    max_length=max_length,
    truncation='only_second',
    stride=stride,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding='max_length')

  offset_mapping = inputs.pop('offset_mapping')
  orig_sample_idxs = inputs.pop('overflow_to_sample_mapping')

  answers = batch['answers']
  start_idxs, end_idxs = [], []

  for i, offset in enumerate(offset_mapping):
    sample_idx = orig_sample_idxs[i]
    answer = answers[sample_idx]

    ans_start_char = answer['answer_start'][0]
    ans_end_char = ans_start_char + len(answer['text'][0])

    sequence_ids = inputs.sequence_ids(i)

    ctx_start = sequence_ids.index(1)
    ctx_end = len(sequence_ids) - sequence_ids[::-1].index(1) - 1

    start_idx, end_idx = find_answer_token_idx(
      ctx_start,
      ctx_end,
      ans_start_char,
      ans_end_char,
      offset
    )

    start_idxs.append(start_idx)
    end_idxs.append(end_idx)

  inputs['start_positions'] = start_idxs
  inputs['end_positions'] = end_idxs
  return inputs

In [48]:
train_dataset = raw_dataset['train'].map(
    tokenize_fn_train,
    batched=True,
    remove_columns=raw_dataset['train'].column_names
)

len(raw_dataset['train']), len(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

(87599, 88729)

In [49]:
raw_dataset['validation'][0]

{'id': '56be4db0acb8001400a502ec',
 'title': 'Super_Bowl_50',
 'context': 'Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi\'s Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.',
 'question': 'Which NFL team represented the AFC at Super Bowl 50?',
 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],


In [50]:
def tokenize_fn_validation(batch):
  questions = [q.strip() for q in batch['question']]

  inputs = tokenizer(
    questions,
    batch['context'],
    max_length=max_length,
    truncation='only_second',
    stride=stride,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding='max_length')

  orig_sample_idxs = inputs.pop('overflow_to_sample_mapping')
  sample_ids = []

  answers = batch['answers']
  start_idxs, end_idxs = [], []

  for i in range(len(inputs['input_ids'])):
    sample_idx = orig_sample_idxs[i]
    sample_ids.append(batch['id'][sample_idx])

    sequence_ids = inputs.sequence_ids(i)
    offset = inputs['offset_mapping'][i]
    inputs['offset_mapping'][i] = [
        x if sequence_ids[j] == 1 else None for j, x in enumerate(offset)]

  inputs['sample_id'] = sample_ids

  return inputs

In [51]:
validation_dataset = raw_dataset['validation'].map(
    tokenize_fn_validation,
    batched=True,
    remove_columns=raw_dataset['validation'].column_names
)

len(raw_dataset['validation']), len(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(10570, 10822)

In [52]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [53]:
import evaluate

In [54]:
metric = evaluate.load('squad')

In [55]:
predicted_answers = [
    { 'id': '1', 'prediction_text': 'Albert Einstein' },
    { 'id': '2', 'prediction_text': 'physicist' },
    { 'id': '3', 'prediction_text': 'general relativity' }
]

true_answers = [
    { 'id': '1', 'answers': { 'text': ['Albert Einstein'], 'answer_start': [100] } },
    { 'id': '2', 'answers': { 'text': ['physicist'], 'answer_start': [100] } },
    { 'id': '3', 'answers': { 'text': ['special relativity'], 'answer_start': [100] } }
]

metric.compute(predictions=predicted_answers, references=true_answers)

{'exact_match': 66.66666666666667, 'f1': 83.33333333333333}

In [56]:
predicted_answers = [
    { 'id': '56be4db0acb8001400a502ec', 'prediction_text': 'Albert Einstein' },
    { 'id': '56be4db0acb8001400a502ed', 'prediction_text': 'physicist' },
    { 'id': '56be4db0acb8001400a502ee', 'prediction_text': 'general relativity' }
]

true_answers = [
    { 'id': '56be4db0acb8001400a502ec', 'answers': { 'text': ['Albert Einstein'], 'answer_start': [100] } },
    { 'id': '56be4db0acb8001400a502ed', 'answers': { 'text': ['physicist'], 'answer_start': [100] } },
    { 'id': '56be4db0acb8001400a502ee', 'answers': { 'text': ['special relativity'], 'answer_start': [100] } }
]

metric.compute(predictions=predicted_answers, references=true_answers)





{'exact_match': 66.66666666666667, 'f1': 83.33333333333333}

In [57]:
small_validation_dataset = raw_dataset['validation'].select(range(100))
trained_checkpoint = 'distilbert-base-cased-distilled-squad'

tokenizer2 = AutoTokenizer.from_pretrained(trained_checkpoint)

old_tokenizer = tokenizer
tokenizer = tokenizer2

small_validation_processed = small_validation_dataset.map(
    tokenize_fn_validation,
    batched=True,
    remove_columns=raw_dataset['validation'].column_names
)

tokenizer = old_tokenizer

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [58]:
import torch
from transformers import AutoModelForQuestionAnswering

In [59]:
small_model_inputs = small_validation_processed.remove_columns(['sample_id', 'offset_mapping'])
small_model_inputs.set_format('torch')

In [60]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [61]:
small_model_inputs_gpu = {
    k: small_model_inputs[k][:].to(device) for k in small_model_inputs.column_names
}

In [62]:
trained_model = AutoModelForQuestionAnswering.from_pretrained(trained_checkpoint).to(device)

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [63]:
with torch.no_grad():
  outputs = trained_model(**small_model_inputs_gpu)

In [64]:
outputs

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ -2.2607,  -5.1783,  -5.2709,  ...,  -9.5243,  -9.5183,  -9.5288],
        [ -2.5961,  -5.5482,  -5.5313,  ...,  -9.9597,  -9.9533,  -9.9860],
        [ -3.7127,  -7.1848,  -8.5388,  ..., -11.6557, -11.6571, -11.6505],
        ...,
        [ -2.0260,  -4.4167,  -4.4980,  ...,  -8.1479,  -8.1530,  -8.1760],
        [ -4.1553,  -5.8304,  -7.1643,  ..., -10.5255, -10.5251, -10.4890],
        [ -3.2000,  -5.8162,  -6.7249,  ...,  -9.4935,  -9.5038,  -9.4871]],
       device='cuda:0'), end_logits=tensor([[ -0.7353,  -4.9236,  -5.1048,  ...,  -8.8734,  -8.8916,  -8.8550],
        [ -1.3056,  -5.3870,  -5.4945,  ...,  -9.4895,  -9.5039,  -9.4958],
        [ -2.7649,  -7.2201,  -9.0916,  ..., -11.3106, -11.3414, -11.2702],
        ...,
        [ -0.0768,  -4.8210,  -4.4374,  ...,  -8.0483,  -8.0502,  -7.9903],
        [ -2.7347,  -5.3650,  -7.2549,  ..., -10.0498, -10.0661,  -9.9886],
        [ -1.0991,  -4.2569,  -6.1267,  ...,  -8

In [65]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()

In [66]:
small_validation_processed['sample_id'][:5]

['56be4db0acb8001400a502ec',
 '56be4db0acb8001400a502ed',
 '56be4db0acb8001400a502ee',
 '56be4db0acb8001400a502ef',
 '56be4db0acb8001400a502f0']

In [67]:
len(validation_dataset['sample_id'])

10822

In [68]:
len(set(validation_dataset['sample_id']))

10570

In [69]:
sample_id2idxs = {}
for i, id_ in enumerate(small_validation_processed['sample_id']):
  if id_ not in sample_id2idxs:
    sample_id2idxs[id_] = [i]
  else:
    print('here')
    sample_id2idxs[id_].append(i)


In [70]:
start_logits.shape, end_logits.shape

((100, 384), (100, 384))

In [71]:
(-start_logits[0]).argsort()

array([ 46,  57,  47,  38,  39,  58,  50,  43,  45,  54,  56,  49,  13,
        42,  40,  35,  27,  31,  48,  41,  53,  44,  37,  59,  78,  15,
         0,  52,  24,  65,  81,  70,  18,  51,  55,  26,  69,  29,  28,
        75,  61,  64,  23,  36,  32,  11, 101,  62,  66,  34,  95,  30,
        63,  21,  19,  20,  17,  14,  22,  33,  68,  87, 171,  12,  76,
        71,  73,  92, 110,  84, 151,   1,  74,   2,   6,  16,  80,  79,
       105,  98,  10,  96, 136, 169, 106, 100,  93, 165,  67, 109,   8,
        90,   3, 115,  60,   5,  97,   7, 103, 102,  86,  72, 111,  89,
       108,   4,  88,  25, 132,  77, 123, 150, 124, 153,  83, 118,  82,
        85, 107, 114, 143, 164, 137, 130, 166, 159, 131,  91,   9, 144,
       139, 160,  94, 141, 128, 112, 134, 152, 170, 154, 117, 127, 104,
       140, 157, 155, 133, 145, 119, 162, 138, 135, 156, 167, 168, 126,
       148, 163, 161, 116,  99, 120, 142, 158, 125, 146, 113, 121, 147,
       149, 129, 122, 311, 312, 304, 309, 313, 310, 300, 307, 31

In [72]:
start_logits[0][(-start_logits[0]).argsort()]

array([10.694441  ,  9.803685  ,  4.4599757 ,  4.4004855 ,  2.9437797 ,
        2.7017365 ,  2.0126479 ,  1.5780729 ,  0.52237177,  0.02073575,
       -0.0280281 , -0.04971485, -0.38572806, -0.69453734, -0.79794735,
       -0.8678019 , -0.8722033 , -1.351686  , -1.3703711 , -1.3878821 ,
       -1.5135074 , -1.735549  , -1.8827065 , -1.8932878 , -1.9078944 ,
       -1.9304932 , -2.2607286 , -2.2983837 , -2.3069303 , -2.502741  ,
       -2.5100617 , -2.5308363 , -2.5399933 , -2.671813  , -2.7323515 ,
       -2.7710161 , -2.771362  , -2.9521313 , -3.060462  , -3.1706033 ,
       -3.2045448 , -3.5693386 , -3.5798054 , -3.6668835 , -3.725059  ,
       -3.7498522 , -3.7632148 , -3.996812  , -4.0113244 , -4.068799  ,
       -4.094482  , -4.1954737 , -4.2383094 , -4.3323603 , -4.352413  ,
       -4.387959  , -4.3886075 , -4.3966107 , -4.679048  , -4.7030287 ,
       -4.7757573 , -4.7778096 , -4.788215  , -4.788247  , -4.822124  ,
       -4.8725367 , -4.8849335 , -4.8981476 , -5.0720954 , -5.10

In [73]:
small_validation_processed['offset_mapping'][0]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 [0, 5],
 [6, 10],
 [11, 13],
 [14, 17],
 [18, 20],
 [21, 29],
 [30, 38],
 [39, 43],
 [44, 46],
 [47, 56],
 [57, 60],
 [61, 69],
 [70, 72],
 [73, 76],
 [77, 85],
 [86, 94],
 [95, 101],
 [102, 103],
 [103, 106],
 [106, 107],
 [108, 111],
 [112, 115],
 [116, 120],
 [121, 127],
 [127, 128],
 [129, 132],
 [133, 141],
 [142, 150],
 [151, 161],
 [162, 163],
 [163, 166],
 [166, 167],
 [168, 176],
 [177, 183],
 [184, 191],
 [192, 200],
 [201, 204],
 [205, 213],
 [214, 222],
 [223, 233],
 [234, 235],
 [235, 238],
 [238, 239],
 [240, 248],
 [249, 257],
 [258, 266],
 [267, 269],
 [269, 270],
 [270, 272],
 [273, 275],
 [276, 280],
 [281, 286],
 [287, 292],
 [293, 298],
 [299, 303],
 [304, 309],
 [309, 310],
 [311, 314],
 [315, 319],
 [320, 323],
 [324, 330],
 [331, 333],
 [334, 342],
 [343, 344],
 [344, 345],
 [346, 350],
 [350, 351],
 [352, 354],
 [355, 359],
 [359, 360],
 [360, 361],
 [362, 369],
 [370, 37

In [74]:
n_largest = 20
max_answer_length = 30
predicted_answers = []

for sample in small_validation_dataset:
  sample_id = sample['id']
  context = sample['context']

  best_score = float('-inf')
  best_answer = None

  for idx in sample_id2idxs[sample_id]:
    start_logit = start_logits[idx]
    end_logit = end_logits[idx]
    offsets = small_validation_processed[idx]['offset_mapping']

    start_indices = (-start_logit).argsort()
    end_indices = (-end_logit).argsort()

    for start_idx in start_indices[:n_largest]:
      for end_idx in end_indices[:n_largest]:

        if offsets[start_idx] is None or offsets[end_idx] is None:
          continue

        if end_idx < start_idx:
          continue

        if end_idx - start_idx + 1 > max_answer_length:
          continue

        score = start_logit[start_idx] + end_logit[end_idx]
        if score > best_score:
          best_score = score

          first_ch = offsets[start_idx][0]
          last_ch = offsets[end_idx][1]

          best_answer = context[first_ch:last_ch]

  predicted_answers.append({ 'id': sample_id, 'prediction_text': best_answer })


In [75]:
small_validation_dataset['answers'][0]

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'],
 'answer_start': [177, 177, 177]}

In [76]:
true_answers = [
    {'id': x['id'], 'answers': x['answers']} for x in small_validation_dataset
    ]

In [77]:
metric.compute(predictions=predicted_answers, references=true_answers)

{'exact_match': 83.0, 'f1': 88.25000000000004}

In [78]:
from tqdm.autonotebook import tqdm

In [79]:
def compute_metrics(start_logits, end_logits, processed_dataset, orig_dataset):
  sample_id2idxs = {}
  for i, id_ in enumerate(processed_dataset['sample_id']):
    if id_ not in sample_id2idxs:
      sample_id2idxs[id_] = [i]
    else:
      sample_id2idxs[id_].append(i)

  predicted_answers = []

  for sample in tqdm(orig_dataset):
    sample_id = sample['id']
    context = sample['context']

    best_score = float('-inf')
    best_answer = None

    for idx in sample_id2idxs[sample_id]:
      start_logit = start_logits[idx]
      end_logit = end_logits[idx]

      offsets = processed_dataset[idx]['offset_mapping']

      start_indices = (-start_logit).argsort()
      end_indices = (-end_logit).argsort()

      for start_idx in start_indices[:n_largest]:
        for end_idx in end_indices[:n_largest]:

          if offsets[start_idx] is None or offsets[end_idx] is None:
            continue

          if end_idx < start_idx:
            continue

          if end_idx - start_idx + 1 > max_answer_length:
            continue

          score = start_logit[start_idx] + end_logit[end_idx]
          if score > best_score:
            best_score = score

            first_ch = offsets[start_idx][0]
            last_ch = offsets[end_idx][1]

            best_answer = context[first_ch:last_ch]

    predicted_answers.append({ 'id': sample_id, 'prediction_text': best_answer })

  true_answers = [
    {'id': x['id'], 'answers': x['answers']} for x in orig_dataset
    ]

  return metric.compute(predictions=predicted_answers, references=true_answers)


In [80]:
compute_metrics(
    start_logits,
    end_logits,
    small_validation_processed,
    small_validation_dataset)

  0%|          | 0/100 [00:00<?, ?it/s]

{'exact_match': 83.0, 'f1': 88.25000000000004}

In [81]:
model = AutoModelForQuestionAnswering.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [82]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    'finetuned-squad',
    eval_strategy='no',
    save_strategy='epoch',
    learning_rate = 2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True
)

In [83]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [84]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator
)

In [85]:
trainer.train()

Step,Training Loss
500,3.317407
1000,2.207299
1500,1.865788
2000,1.666943
2500,1.620511
3000,1.524658
3500,1.467998
4000,1.412944
4500,1.382430
5000,1.384541


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=33276, training_loss=1.0304112228611826, metrics={'train_runtime': 3357.8979, 'train_samples_per_second': 79.272, 'train_steps_per_second': 9.91, 'total_flos': 2.608361755366349e+16, 'train_loss': 1.0304112228611826, 'epoch': 3.0})

In [86]:
trainer_output = trainer.predict(validation_dataset)

In [87]:
type(trainer_output)

transformers.trainer_utils.PredictionOutput

In [88]:
trainer_output

PredictionOutput(predictions=(array([[ -7.0351562, -11.2578125, -11.4921875, ..., -11.65625  ,
        -11.671875 , -11.640625 ],
       [ -7.265625 , -11.265625 , -11.5078125, ..., -11.640625 ,
        -11.65625  , -11.6328125],
       [ -7.5234375, -11.203125 , -11.34375  , ..., -11.640625 ,
        -11.6484375, -11.65625  ],
       ...,
       [ -4.7929688, -11.3828125, -11.484375 , ..., -11.796875 ,
        -11.84375  , -11.8125   ],
       [ -3.140625 ,  -9.890625 ,  -9.234375 , ..., -11.734375 ,
        -11.7578125, -11.6796875],
       [ -4.6640625, -11.203125 , -11.34375  , ..., -11.765625 ,
        -11.8125   , -11.78125  ]], dtype=float32), array([[ -6.4882812, -11.265625 , -11.046875 , ..., -11.859375 ,
        -11.859375 , -11.8828125],
       [ -6.7929688, -11.2421875, -11.0390625, ..., -11.859375 ,
        -11.859375 , -11.8828125],
       [ -7.0507812, -11.5234375, -11.7421875, ..., -11.9609375,
        -11.953125 , -11.9296875],
       ...,
       [ -4.0703125, -11.3828

In [89]:
predictions, _, _ = trainer_output

In [90]:
predictions

(array([[ -7.0351562, -11.2578125, -11.4921875, ..., -11.65625  ,
         -11.671875 , -11.640625 ],
        [ -7.265625 , -11.265625 , -11.5078125, ..., -11.640625 ,
         -11.65625  , -11.6328125],
        [ -7.5234375, -11.203125 , -11.34375  , ..., -11.640625 ,
         -11.6484375, -11.65625  ],
        ...,
        [ -4.7929688, -11.3828125, -11.484375 , ..., -11.796875 ,
         -11.84375  , -11.8125   ],
        [ -3.140625 ,  -9.890625 ,  -9.234375 , ..., -11.734375 ,
         -11.7578125, -11.6796875],
        [ -4.6640625, -11.203125 , -11.34375  , ..., -11.765625 ,
         -11.8125   , -11.78125  ]], dtype=float32),
 array([[ -6.4882812, -11.265625 , -11.046875 , ..., -11.859375 ,
         -11.859375 , -11.8828125],
        [ -6.7929688, -11.2421875, -11.0390625, ..., -11.859375 ,
         -11.859375 , -11.8828125],
        [ -7.0507812, -11.5234375, -11.7421875, ..., -11.9609375,
         -11.953125 , -11.9296875],
        ...,
        [ -4.0703125, -11.3828125, -11.

In [91]:
start_logits, end_logits = predictions

In [92]:
compute_metrics(
    start_logits,
    end_logits,
    validation_dataset,
    raw_dataset['validation'])

  0%|          | 0/10570 [00:00<?, ?it/s]

{'exact_match': 77.360454115421, 'f1': 85.40299908964742}

In [93]:
trainer.save_model('my_saved_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [94]:
from transformers import pipeline

qa = pipeline('question-answering', model='my_saved_model', device=0)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [95]:
context = 'Today I went to the store to purchase a carton of milk'
question = 'What did I buy'

In [96]:
qa(context=context, question=question)

{'score': 0.6800779104232788,
 'start': 38,
 'end': 54,
 'answer': 'a carton of milk'}